# Week 4 - Part 1: Package the model and prove it is served correctly

Author: Muhammad Raheel Ijaz | Course: Introduction to Applied AI | Project 1: ML-Powered Customer Analytics

## Project Overview

Run this notebook from the project folder (`D:\AI\customer-churn-prediction`) with the **venv** kernel.
It does **not** retrain anything: it loads the `churn_model.joblib` and `churn_model_metadata.json`
that `week3_optimization.ipynb` already saved, so no *Run All* of Week 3 is needed.
Run the cells top to bottom.

### Cell 0 - Rebuild `df`, `build_features`, `X` exactly as Week 3 did (copied from its Task 1.2)

In [1]:
import json, joblib, sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

df = pd.read_csv('customer_data.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

def build_features(data):
    X_ = data.drop(columns=['customerID', 'Churn'])
    return pd.get_dummies(X_, drop_first=True).astype(float)

y = (df['Churn'] == 'Yes').astype(int)
X = build_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(df.shape, X.shape, X_train.shape, X_test.shape)

(7043, 21) (7043, 30) (5634, 30) (1409, 30)


### Cell 1 - Task 1.1: What exactly did we save?

In [2]:
model = joblib.load('churn_model.joblib')
cols = list(model.feature_names_in_)          # exact training columns, in order
final = model[-1] if hasattr(model, 'steps') else model
print('Model          :', type(final).__name__)
print('Feature columns:', len(cols))
print('scikit-learn   :', sklearn.__version__)   # you will pin this version
try:
    import xgboost
    print('xgboost        :', xgboost.__version__)   # pin this too (final model is XGBoost)
except ImportError:
    print('xgboost        : not installed')
assert cols == list(X.columns), 'Saved model columns differ from rebuilt X columns!'
print('Column order matches the rebuilt training matrix.')

Model          : XGBClassifier
Feature columns: 30
scikit-learn   : 1.9.0
xgboost        : 3.3.0
Column order matches the rebuilt training matrix.


### Cell 2 - Task 1.2: The serving-time encoder

In [3]:
RAW_COLS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
            'TotalCharges']

def prepare_input(raw, columns):
    """Turn raw customer rows into exactly the columns the model saw."""
    X_ = pd.get_dummies(raw[RAW_COLS])            # NO drop_first when serving
    return X_.reindex(columns=columns, fill_value=0).astype(float)

### Cell 3 - Task 1.3: Reproduce the training-serving skew bug

In [4]:
one = df[df['Contract'] == 'Two year'].head(1)
wrong = build_features(one).reindex(columns=cols, fill_value=0)   # Week 3 way, one row
right = prepare_input(one, cols)

print('Contract columns, wrong:', wrong.filter(like='Contract').values)
print('Contract columns, right:', right.filter(like='Contract').values)
print(f'P(churn) wrong: {model.predict_proba(wrong)[0, 1]:.3f}')
print(f'P(churn) right: {model.predict_proba(right)[0, 1]:.3f}')

Contract columns, wrong: [[0 0]]
Contract columns, right: [[0. 1.]]
P(churn) wrong: 0.137
P(churn) right: 0.011


### Cell 4 - Task 1.4: The parity test (every customer)

In [5]:
raw = df.drop(columns=['Churn'])
served = prepare_input(raw, cols)
p_train = model.predict_proba(X[cols])[:, 1]   # how the model was evaluated
p_serve = model.predict_proba(served)[:, 1]    # how the app will call it
print('Largest difference:', np.abs(p_train - p_serve).max())
assert np.allclose(p_train, p_serve), 'Training-serving skew detected!'
print(f'Parity test passed on {len(raw)} customers')

Largest difference: 0.0
Parity test passed on 7043 customers


### Cell 5 - Task 1.5: What-if analysis

In [6]:
def what_if(model, row, columns, changes):
    base = model.predict_proba(prepare_input(row, columns))[0, 1]
    rows = []
    for feature, value in changes:
        alt = row.copy()
        alt[feature] = value
        p = model.predict_proba(prepare_input(alt, columns))[0, 1]
        rows.append({'change': f'{feature} = {value}',
                     'p_churn': round(p, 3), 'delta': round(p - base, 3)})
    return base, pd.DataFrame(rows)

row = raw.iloc[[int(np.argmax(p_serve))]]      # the riskiest customer
base, table = what_if(model, row, cols, [
    ('Contract', 'One year'), ('Contract', 'Two year'),
    ('PaymentMethod', 'Credit card (automatic)')])
print(f'Baseline P(churn) = {base:.3f}')
print(table.to_string(index=False))

Baseline P(churn) = 0.942
                                 change  p_churn  delta
                    Contract = One year    0.883 -0.059
                    Contract = Two year    0.794 -0.148
PaymentMethod = Credit card (automatic)    0.912 -0.030


### Cell 5b - Choose the contact threshold from a cost analysis (training data only)

The lab's model card uses **FN = PKR 6,000** (a churner we miss) and **FP = PKR 1,000** (a wasted retention contact).
We use *out-of-fold* predictions on the **training** set only, so the test set stays untouched.
If you already have a threshold from your own Week 2 cost analysis, set `THRESHOLD` by hand in Cell 6 instead.

In [7]:
FN_COST, FP_COST = 6000, 1000
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(clone(final), X_train, y_train, cv=cv, method='predict_proba')[:, 1]

rows = []
for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
    flagged = oof >= t
    fn = int(((~flagged) & (y_train == 1)).sum())
    fp = int((flagged & (y_train == 0)).sum())
    rows.append({'threshold': t, 'flagged_%': round(100 * flagged.mean(), 1),
                 'FN': fn, 'FP': fp, 'total_cost_PKR': fn * FN_COST + fp * FP_COST})
cost = pd.DataFrame(rows)
print(cost.to_string(index=False))
best_t = float(cost.loc[cost['total_cost_PKR'].idxmin(), 'threshold'])
print('\nCost-minimising threshold on the 0.05 grid:', best_t)
print('Theory for calibrated probabilities: FP/(FP+FN) =', round(FP_COST / (FP_COST + FN_COST), 3))

 threshold  flagged_%   FN   FP  total_cost_PKR
      0.05       72.5   35 2624         2834000
      0.10       60.9   96 2032         2608000
      0.15       53.5  145 1664         2534000
      0.20       47.6  198 1386         2574000
      0.25       42.4  270 1165         2785000
      0.30       37.1  357  954         3096000
      0.35       32.5  433  770         3368000
      0.40       27.9  529  607         3781000
      0.45       24.7  600  494         4094000
      0.50       21.3  694  398         4562000
      0.55       17.7  804  309         5133000
      0.60       14.3  915  227         5717000
      0.65       11.4 1023  168         6306000
      0.70        8.4 1122   99         6831000
      0.75        5.7 1224   48         7392000
      0.80        3.3 1328   19         7987000
      0.85        1.2 1435    7         8617000
      0.90        0.2 1483    1         8899000
      0.95        0.0 1495    0         8970000

Cost-minimising threshold on the 0.05 g

### Cell 6 - Task 1.6: Save `model_meta.json` and `sample_customers.csv`

Numbers come straight from `churn_model_metadata.json`, which Week 3 wrote in the same run as `churn_model.joblib`,
so nothing is typed by hand. The check below re-scores the held-out test set only to prove the joblib matches that JSON
(an integrity check, not a model-selection step).

In [8]:
with open('churn_model_metadata.json') as f:
    m = json.load(f)
print({k: v for k, v in m.items() if k != 'features'})

check_auc = roc_auc_score(y_test, model.predict_proba(X_test[cols])[:, 1])
assert abs(check_auc - m['test_auc']) < 1e-6, (
    f'joblib test AUC {check_auc:.6f} != JSON {m["test_auc"]:.6f}: '
    'files come from different runs, re-run Week 3 once and retry.')
print('churn_model.joblib matches churn_model_metadata.json')

THRESHOLD = best_t            # cost-minimising value from Cell 5b. You must be able to defend it:
                              # check 'flagged_%' (how many customers the team must call) and change it
                              # by hand (a multiple of 0.05, e.g. 0.30) if that is not realistic.
print(f'Chosen threshold {THRESHOLD}: flags about '
      f"{cost.loc[cost['threshold'] == THRESHOLD, 'flagged_%'].iloc[0]}% of training customers")

meta = {
    'model_name': type(final).__name__,
    'sklearn_version': sklearn.__version__,
    'feature_columns': cols,
    'threshold': THRESHOLD,
    'cv_auc': round(m['cv_auc_mean'], 4),
    'cv_auc_std': round(m['cv_auc_std'], 4),
    'test_auc': round(m['test_auc'], 4),
    'training_data': 'IBM Telco Customer Churn, 7,043 customers',
    'version': '1.0',
}
try:
    meta['xgboost_version'] = xgboost.__version__
except NameError:
    pass
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved model_meta.json with', len(cols), 'columns')

sample = df.drop(columns=['Churn']).sample(50, random_state=1)
sample.to_csv('sample_customers.csv', index=False)
print('Saved 50 customers for the batch demo')

{'model': 'XGBoost (tuned)', 'cv_auc_mean': 0.850391792513315, 'cv_auc_std': 0.012093255445132794, 'test_auc': 0.8472487018522823, 'test_recall': 0.5267379679144385, 'test_precision': 0.6501650165016502, 'test_accuracy': 0.7991483321504613, 'n_features': 30}
churn_model.joblib matches churn_model_metadata.json
Chosen threshold 0.15: flags about 53.5% of training customers
Saved model_meta.json with 30 columns
Saved 50 customers for the batch demo
